# Trippelintegraler i Python (Scipy og Sympy)

Vi har tidligere sett hvordan Scipy (`quad`, `dblquad`) og Sympy (`sp.integrate`) regner ut enkelt- og dobbeltintegraler -- se `F4_Integrasjon_Scipy_1D2D.ipynb` og `F4_Integrasjon_Sympy_1D2D.ipynb`. Her tar vi det siste steget, opp i **3 dimensjoner**: `scipy.integrate.tplquad` og `sp.integrate` med tre variabler generaliserer det vi allerede kan, helt direkte.

Dette er også en fin sjekk på notebooken `F8_integrasjon_Delaunay_2Dog3D.ipynb`, der vi tilnærmet volum og trippelintegraler geometrisk ved å dele opp i tetraedre. Her regner vi ut de samme integralene igjen, numerisk presist (Scipy) eller helt eksakt (Sympy), slik at vi har noe å sjekke de geometriske tilnærmingene opp mot.

# Scipy for 3D

Utregning av trippelintegralet
$$\int_a^b\left(\int_{g(x)}^{h(x)}\left(\int_{q(x,y)}^{r(x,y)} f(z,y,x)\, dz\right) dy\right) dx$$
gjøres med `tplquad(f, a, b, gfun, hfun, qfun, rfun)`. Dette er en direkte generalisering av `dblquad`:

* `f` tar argumentene sine "innenfra og ut", akkurat som i `dblquad`: `f(z, y, x)` -- $z$ først, $x$ sist.
* `gfun`, `hfun` er funksjoner av $x$ som gir $y$-grensene (samme som i `dblquad`).
* `qfun`, `rfun` er **nye**: funksjoner av $(x,y)$ (i den rekkefølgen) som gir $z$-grensene.
* Er grensene konstante, bruker vi bare tall i stedet for funksjoner -- akkurat som i `dblquad`.

## Eksempel

La oss regne ut det samme integralet som vi tilnærmet med håndplukkede og automatiske tetraedre i forrige notebook:
$$\int_0^2\int_0^1\int_0^1 (x^2+y+z)\, dz\, dy\, dx.$$

In [1]:
import numpy as np
from scipy import integrate

f = lambda z, y, x: x**2 + y + z   # legg merke til rekkefølgen: z (innerst), y, x (ytterst)

result = integrate.tplquad(f, 0, 2, 0, 1, 0, 1)   # x fra 0 til 2, y fra 0 til 1, z fra 0 til 1

print(result)
print(result[0], "er her integralet evaluert, mens") # value of the integral
print(result[1], "er feilestimatet") # estimate of error
print("Eksakt svar (14/3):", 14/3)


(4.666666666666666, 6.084550745299033e-14)
4.666666666666666 er her integralet evaluert, mens
6.084550745299033e-14 er feilestimatet
Eksakt svar (14/3): 4.666666666666667


### Oppgave (løses i forelesningen)

Bruk Scipy sin `tplquad` til å regne ut
$$\int_0^\pi\int_0^1\int_0^2 \bigl(\sin(x)+y+z\bigr)\, dz\, dy\, dx.$$

In [2]:
# Skriv koden her


## Variable grenser: volumet av et tetraeder

Akkurat som for `dblquad` kan grensene avhenge av de andre variablene -- vi bytter da bare ut tallene med funksjoner. Som eksempel regner vi ut volumet av enhets-tetraederet (simpleks) $0\le x\le 1,\ 0\le y\le 1-x,\ 0\le z\le 1-x-y$:
$$\int_0^1\int_0^{1-x}\int_0^{1-x-y} 1\, dz\, dy\, dx.$$

Legg merke til at dette er nøyaktig samme tetraeder-form (og samme volum, $\tfrac16$) som hvert av de $6$ tetraedrene vi delte enhetskuben i, i forrige notebook.

In [3]:
f = lambda z, y, x: 1

g = lambda x: 0            # nedre grense for y
h = lambda x: 1 - x        # øvre grense for y
q = lambda x, y: 0         # nedre grense for z
r = lambda x, y: 1 - x - y # øvre grense for z

result = integrate.tplquad(f, 0, 1, g, h, q, r)
print(result)
print("Eksakt svar (1/6):", 1/6)


(0.16666666666666669, 1.1054067417904425e-14)
Eksakt svar (1/6): 0.16666666666666666


## Lite (og litt vrient) triks: volumet av en kule

Akkurat som i 2D-notebooken kan vi regne ut et integral over et krumt område ved å integrere over en boks som inneholder området, og sette funksjonen lik $0$ utenfor. Vi prøver oss på volumet av enhetskulen (som vi også så på i forrige notebook, med Delaunay):
$$V = \iiint_B 1\, dV, \qquad B = \{(x,y,z) : x^2+y^2+z^2 \le 1\}.$$

I 3D er denne typen "hakkete" (diskontinuerlige) funksjon mye vanskeligere for `tplquad` å håndtere enn i 2D: den adaptive metoden prøver å finne den krumme randen ved å dele opp i stadig flere og mindre delintervaller, og vi må løsne litt på toleransen (`epsabs`, `epsrel`) for at utregningen skal bli ferdig i rimelig tid -- selv da får vi en advarsel. Dette er nettopp grunnen til at vi i forrige notebook heller brukte en punktsky + Delaunay-tetraedrisering for slike krumme områder.

In [4]:
def f(z, y, x):
    return 1.0 if x*x + y*y + z*z <= 1 else 0.0

result = integrate.tplquad(f, -1, 1, -1, 1, -1, 1, epsabs=1e-3, epsrel=1e-3)
print(result)
print("Eksakt svar (4*pi/3):", 4*np.pi/3)


/usr/local/lib/python3.11/dist-packages/scipy/integrate/_quadpack_py.py:1286: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  quad_r = quad(f, low, high, args=args, full_output=self.full_output,


(4.188908768026616, 0.007062504840002916)
Eksakt svar (4*pi/3): 4.1887902047863905


# Sympy for 3D

Sympy generaliserer like direkte: vi gir `sp.integrate` tre `(variabel, nedre, øvre)`-par i stedet for to,
$$\int_a^b\int_c^d\int_e^h f(x,y,z)\, dx\, dy\, dz.$$

## Eksempel

Vi regner ut det samme boks-integralet som over, men nå symbolsk:

In [5]:
import sympy as sp

x, y, z = sp.symbols('x y z')
f = x**2 + y + z

svar = sp.integrate(f, (z, 0, 1), (y, 0, 1), (x, 0, 2))
print(svar)
print(float(svar))


14/3
4.666666666666667


### Oppgave (løses i forelesningen)

Bruk Sympy til å regne ut, symbolsk og numerisk,
$$\int_0^1\int_0^2\int_0^3 x^2 y\, dz\, dy\, dx.$$

In [6]:
# Skriv koden her


## Variable grenser: samme tetraeder, symbolsk

Vi kan gjenta tetraeder-eksemplet fra Scipy-delen symbolsk -- da får vi et eksakt svar direkte, uten numerisk feil:
$$\int_0^1\int_0^{1-x}\int_0^{1-x-y} 1\, dz\, dy\, dx.$$

In [7]:
svar = sp.integrate(1, (z, 0, 1-x-y), (y, 0, 1-x), (x, 0, 1))
print(svar)


1/6


**Til ettertanke**: Hva skjer om vi bytter rekkefølgen på de tre parene, for eksempel til `(x, 0, 1-y-z), (y, 0, 1-z), (z, 0, 1)`?

### Oppgave (løses i forelesningen)

I forrige notebook regnet dere ut, med et rutenett av punkter og Delaunay,
$$\int_0^1\int_0^2\int_0^1 xyz\, dz\, dy\, dx.$$
Bruk Sympy til å finne det eksakte svaret, og sammenlign med tilnærmingen dere fikk da.

In [8]:
# Skriv koden her


## Oppsummering

1. `tplquad(f, a, b, gfun, hfun, qfun, rfun)` generaliserer `dblquad` direkte: ett ekstra grense-funksjonspar, og `f` tar argumentene sine innenfra og ut: `f(z, y, x)`.
2. `sp.integrate` generaliserer like direkte -- bare gi den tre `(variabel, nedre, øvre)`-par i stedet for to.
3. Grensene kan avhenge av de andre variablene i begge biblioteker, akkurat som i 2D: vi bytter da bare ut tall med funksjoner (Scipy) eller uttrykk (Sympy).
4. Adaptiv numerisk kvadratur (`tplquad`) sliter med diskontinuerlige/krumme rand-integraler i 3D -- der er punktsky + Delaunay-tetraedrisering (forrige notebook) eller Monte Carlo ofte et bedre valg.
5. Disse metodene gir eksakte eller svært presise svar, og er nyttige nettopp for å sjekke de geometriske tilnærmingene fra Delaunay-notebooken opp mot.